In [1]:
from pathlib import Path
import pandas as pd
import polars as pl
 
from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.features.build_dataset import assemble_feature_matrix, gbm_features
from credit_risk.features.woe import WOEEncoder, rank_features_by_iv, prune_correlated_features
from credit_risk.evaluation.diagnostics import coefficient_sign_report, multicollinearity_report, drop_until_signs_are_clean
from sklearn.linear_model import LogisticRegression

In [2]:
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
df = load_raw_accepted_loans(DATA_PATH)
labeled = build_target(df)
final = assemble_feature_matrix(labeled, Path("../configs/base.yaml"))
train = final.filter(pl.col("split") == "train")

print(train.shape)

(453809, 161)


### Candidate Pool

In [3]:
candidates = gbm_features(final)
print(f"{len(candidates)} candidate features")

76 candidate features


### IV Ranking

In [4]:
iv_ranked = rank_features_by_iv(train, candidates, n_bins=10).sort("iv", descending=True)
print(iv_ranked.to_pandas().to_string(index=False))
 
above_threshold = iv_ranked.filter(pl.col("iv") >= 0.02)["feature"].to_list()
print(f"{len(above_threshold)} features with IV >= 0.02")

                           feature       iv   strength
                         sub_grade 0.425872     strong
                             grade 0.395117     strong
                          int_rate 0.391062     strong
                       term_months 0.195354     medium
                    fico_range_low 0.111152     medium
                               dti 0.054896       weak
              acc_open_past_24mths 0.054333       weak
                    bc_open_to_buy 0.048846       weak
               verification_status 0.041567       weak
                num_tl_op_past_12m 0.039743       weak
                        annual_inc 0.036686       weak
                       avg_cur_bal 0.033784       weak
                  percent_bc_gt_75 0.033640       weak
                    total_bc_limit 0.033454       weak
                   tot_hi_cred_lim 0.030590       weak
                    mo_sin_rcnt_tl 0.029839       weak
                           bc_util 0.029348       weak
          

### Preliminary fit + diagnostics (exploratory - this model is thrown away)

In [5]:
encoder = WOEEncoder(features=above_threshold, n_bins=10).fit(train)
train_woe = encoder.transform(train)
woe_cols = [f"{f}_woe" for f in above_threshold]
prelim_model = LogisticRegression(max_iter=1000).fit(
    train_woe.select(woe_cols).to_pandas(), train_woe["default_flag"].to_pandas()
)
 
print("Coefficient signs (all should be negative - see WOEEncoder docstring for why):")
print(coefficient_sign_report(prelim_model, above_threshold).to_string(index=False))
 
print("Multicollinearity (|correlation| > 0.6):")
collinearity = multicollinearity_report(train_woe, above_threshold, threshold=0.6)
print(collinearity.to_string(index=False) if len(collinearity) else "none found")

Coefficient signs (all should be negative - see WOEEncoder docstring for why):
              feature  coefficient                              flag
           annual_inc      -1.1829                                  
            sub_grade      -0.8579                                  
            loan_amnt      -0.7006                                  
          term_months      -0.5686                                  
 acc_open_past_24mths      -0.5313                                  
       inq_last_6mths      -0.5206                                  
             mort_acc      -0.4773                                  
              purpose      -0.4506                                  
 mths_since_recent_bc      -0.4348                                  
                  dti      -0.2756                                  
       bc_open_to_buy      -0.2444                                  
       fico_range_low      -0.1870                                  
       total_bc_limit   

### Prune correlated features

In [6]:
corr_for_pruning = train_woe.select(woe_cols).to_pandas()
corr_for_pruning.columns = [c.replace("_woe", "") for c in corr_for_pruning.columns]
final_features = prune_correlated_features(above_threshold, corr_for_pruning.corr(), threshold=0.6)
print(f"{len(final_features)} final features (was {len(above_threshold)})")
print(final_features)

17 final features (was 29)
['sub_grade', 'term_months', 'fico_range_low', 'dti', 'acc_open_past_24mths', 'bc_open_to_buy', 'verification_status', 'annual_inc', 'avg_cur_bal', 'percent_bc_gt_75', 'mo_sin_rcnt_tl', 'loan_amnt', 'mths_since_recent_inq', 'num_rev_tl_bal_gt_0', 'purpose', 'mths_since_recent_bc', 'mort_acc']


### Re-check after pruning

In [7]:
clean_features, final_encoder, final_model = drop_until_signs_are_clean(final_features, train)
print(f"\n{len(clean_features)} features after sign-based refinement (was {len(final_features)})")
 
final_train_woe = final_encoder.transform(train)
print("\nMulticollinearity (should still be empty):")
print(multicollinearity_report(final_train_woe, clean_features, threshold=0.6))
print("\nCoefficient signs (should ALL be negative):")
print(coefficient_sign_report(final_model, clean_features).to_string(index=False))

dropping 'percent_bc_gt_75' (coefficient 0.0244, still positive)

16 features after sign-based refinement (was 17)

Multicollinearity (should still be empty):
Empty DataFrame
Columns: [feature_a, feature_b, correlation]
Index: []

Coefficient signs (should ALL be negative):
              feature  coefficient flag
           annual_inc      -1.1543     
            sub_grade      -0.6332     
            loan_amnt      -0.5937     
          term_months      -0.5591     
 acc_open_past_24mths      -0.5351     
              purpose      -0.4708     
             mort_acc      -0.3696     
 mths_since_recent_bc      -0.2892     
                  dti      -0.2670     
          avg_cur_bal      -0.2146     
       fico_range_low      -0.1954     
       bc_open_to_buy      -0.1830     
  num_rev_tl_bal_gt_0      -0.1603     
  verification_status      -0.1341     
mths_since_recent_inq      -0.0706     
       mo_sin_rcnt_tl      -0.0394     
